In [1]:
import pandas as pd
import importlib
import os
from dotenv import load_dotenv
load_dotenv()
import sys
sys.path.append("../")
import process as p

Comme nous l'avons vu dans la partie [1_Extraction](https://github.com/ismaila-b-cisse/portfolio_dataScience/blob/master/1_extraction/main2.ipynb), nous avons un premier dataset (jeux de données) de modèles des marques de nos données e-commerces, notamment temu et darty, issu de l'extraction des modèles et leurs marques sur le site moviles. Cependant, il y a certaines marques qui ne sont pas présentes sur le site. 

Dans ce module de la partie préparation des données, nous allons traiter ces données supplémentaires pour voir s'il y a lieu de faire des nettoyages et transformations avant de les utiliser avec nos données e-commerces.

Avec ces nouvelles données, nous allons construire un référentiel de modèles des marques pour nos données e-commerces.


In [2]:
moviles_df = pd.read_csv('../data/extracted_data/extracted_moviles_data.csv')
# On affiche quelques lignes des données
moviles_df.head()

,marque,modele,date
0,apple,NaN,06-12-2025 18:05:19
1,apple,NaN,06-12-2025 18:05:19
2,apple,Apple iPhone 16,06-12-2025 18:05:19
3,apple,NaN,06-12-2025 18:05:19
4,apple,Apple iPhone 16 Pro,06-12-2025 18:05:19


On peut ici avoir un premier apperçu des données extraites sur le site moviles. On peut voir sur les colonnes les variables 'marque', 'modele' et la date d'extraction.  
Nous avons les cinq premières observations dont toutes les marques sont 'apple', car lors de l'extraction, nous avons trié les marques par ordre alphabétique.
    
On constate à premier vue que la variable modèle a des valeurs nulles. Étant donné que le site moviles classe apparemment les modèles des marques du plus récent au plus anciens, on peut constater que le modèle le plus récent pour la marque 'apple' sur le site est iPhone 16. Or, il y a un modèle plus récent de la marque qui est iPhone 17. Donc, nous aurons besoin de compléter ce dataset des données d'autres sources, peut-être, les site même des marques pour pouvoir récupérer les modèles les plus récents susceptibles d'être dans nos données e-commerces.

In [3]:
# Suppression de la date de l'extraction
moviles_df.drop(['date'], axis=1, inplace=True)
moviles_df.head()

,marque,modele
0,apple,NaN
1,apple,NaN
2,apple,Apple iPhone 16
3,apple,NaN
4,apple,Apple iPhone 16 Pro


# Inspection

In [4]:
moviles_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5106 entries, 0 to 5105
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   marque  5106 non-null   object
 1   modele  4621 non-null   object
dtypes: object(2)
memory usage: 79.9+ KB


In [5]:
moviles_df.describe()

,marque,modele
count,5106,4621
unique,21,4615
top,samsung,Huawei U8180 IDEOS X1
freq,1610,2


Dans ce dataset, nous avons 5106 observations. 
La variable 'marque' a 5106 valeurs, n'a pas donc pas de valeurs nulles et elle a 21 marques distinctes.  
La variable 'modele' a 4621 valeurs, donc contient des valeurs nulles comme nous l'avons déjà constaté ci-dessus. Elle a 4615 modèles qui sont uniques, donc il y a des valeurs qui sont dupliquées 

Voyons d'abord les marques uniques que nous avons dans le dataset, ensuite traitons les valeurs manquantes et enfin les valeurs dupliquées

In [6]:
moviles_df['marque'].unique()

array(['apple', 'asus', 'beafon', 'blackview', 'crosscall', 'cubot',
       'doogee', 'doro', 'google', 'honor', 'huawei', 'motorola',
       'olympia', 'oneplus', 'oppo', 'oukitel', 'realme', 'samsung',
       'vivo', 'xgody', 'xiaomi'], dtype=object)

Nous avons 21 marques distinctes dans notre dataset. Pour rappel, voici les marques qui se trouvent dans nos données initiales (notamment darty et temu) et pour lesquelles nous voulons extraire les modèles dans le module moviles_scraping :
    
    ['samsung', 'apple', 'xiaomi', 'blackview', 'reborn', 'oppo', 'google', 'oukitel', 
    'honor', 'oneplus', 'artfone', 'oscal', 'fossibot', 'cubot', 'nubia', 'doogee', 
    'doro', 'vivo', 'huawei', 'hotwav', 'crosscall', 'lagoona', 'olympia', 'generique', 
    'asus', 'motorola', 'beafon', 'realme', 'iiif150', 'viqee', 'fvh', 'xgody', 
    'rainbuvvy', 'astarry']

Cependant, comme nous avons pu le constater dans les résultats de l'extraction, ces marques suivantes
    
    ["artfone", "astarry", "fossibot", "fvh", "generique", "hotwav", 
    "iiif150", "lagoona", "nubia", "oscal", "rainbuvvy", "reborn", "viqee"] 

ne sont pas présentes sur le site moviles.

Par conséquent, nous devons trouver d'autres sources pour avoir leurs modèles comme pour les modèles récents de toutes les marques ou certaines marques présentes dans notre dataset. 

In [7]:
moviles_df.isnull().sum()

marque      0
modele    485
dtype: int64

Nous avons 485 valeurs nulles dans les modèles. Nous allons les supprimer. On peut se demander comment peut-on avoir ces valeurs nulles alors que ce sont les modèles qui sont listés, donc pourquoi mettre un modèle qui n'a pas de nom ? En réalité, ces champs ne sont pas liés à la présence ou non du modèle, mais à des publicités qui se trouvent entre un modèle et un autre sur le site moviles. 
C'est pourquoi il est inutile de chercher à remplir ces valeurs nulles. Donc, on les supprime.

In [8]:
moviles_df = moviles_df.dropna(ignore_index=True)
moviles_df.isnull().sum() # on n'a plus de valeurs nulles
moviles_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4621 entries, 0 to 4620
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   marque  4621 non-null   object
 1   modele  4621 non-null   object
dtypes: object(2)
memory usage: 72.3+ KB


Après suppression des valeurs nulles, nous avons maintenant 4621 entrées. Maintenant, nous allons traiter les observations dubliquées.

In [9]:
moviles_df.describe()

,marque,modele
count,4621,4621
unique,21,4615
top,samsung,Huawei U8180 IDEOS X1
freq,1463,2


In [10]:
print("Nombre de lignes dupliquées : ", moviles_df.duplicated().sum())

Nombre de lignes dupliquées :  6


Comme on peut bien le constater, il y a 6 lignes qui sont dupliquées. Nous allons voir quelles sont ces lignes.

In [11]:
moviles_df[moviles_df.duplicated(keep=False)]

,marque,modele
1165,huawei,Huawei U8180 IDEOS X1
1220,huawei,Huawei U8180 IDEOS X1
1224,huawei,Huawei U8510 IDEOS X3
1225,huawei,Huawei U8510 IDEOS X3
1386,motorola,Motorola One Zoom
1387,motorola,Motorola One Zoom
2793,samsung,Samsung Galaxy W
2794,samsung,Samsung Galaxy W
3013,samsung,Samsung Galaxy R
3021,samsung,Samsung Galaxy R


In [12]:
# Nous allons conserver les premières lignes et
# supprimer les lignes dupliquées
moviles_df[moviles_df.duplicated()]

,marque,modele
1220,huawei,Huawei U8180 IDEOS X1
1225,huawei,Huawei U8510 IDEOS X3
1387,motorola,Motorola One Zoom
2794,samsung,Samsung Galaxy W
3021,samsung,Samsung Galaxy R
4261,xgody,Xgody X25


Nous allons supprimer ces duplicats

In [13]:
moviles_df = moviles_df.drop_duplicates(ignore_index=True)
moviles_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4615 entries, 0 to 4614
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   marque  4615 non-null   object
 1   modele  4615 non-null   object
dtypes: object(2)
memory usage: 72.2+ KB


In [14]:
moviles_df.describe()

,marque,modele
count,4615,4615
unique,21,4615
top,samsung,Apple iPhone 16
freq,1461,1


À présent, nous pouvons voir qu'il n'y a plus de valeurs nulles dans la variables modèles, non plus d'observation dupliquée dans le dataset.

In [15]:
moviles_df.head()

,marque,modele
0,apple,Apple iPhone 16
1,apple,Apple iPhone 16 Pro
2,apple,Apple iPhone 16 Pro Max
3,apple,Apple iPhone 16 Plus
4,apple,Apple iPhone 15


In [16]:
moviles_df.index

RangeIndex(start=0, stop=4615, step=1)

In [17]:
moviles_df['modele'].unique()

array(['Apple iPhone 16', 'Apple iPhone 16 Pro',
       'Apple iPhone 16 Pro Max', ..., 'Xiaomi MI-2a', 'Xiaomi MI-2',
       'Xiaomi MI-1s'], shape=(4615,), dtype=object)

Nous allons maintenant supprimer le nom de la marque dans les valeurs de la variable 'modele'

In [18]:
indexes = moviles_df.index
p.delete_brand_label(moviles_df, 'marque', 'modele', indexes)

In [19]:
moviles_df.head(5)

,marque,modele
0,apple,iphone 16
1,apple,iphone 16 pro
2,apple,iphone 16 pro max
3,apple,iphone 16 plus
4,apple,iphone 15


In [20]:
moviles_df['modele'].unique()

array(['iphone 16', 'iphone 16 pro', 'iphone 16 pro max', ..., 'mi-2a',
       'mi-2', 'mi-1s'], shape=(4437,), dtype=object)

Maintenant, nous allons compléter ce dataset par d'autres données sur les marques qui n'y sont pas présentes, mais aussi les modèles récents des marques qui y sont présentes, si nécessaire.

Commençons d'abord par compléter les modèles des marques qu'on a déjà. Pour cela, affichons les premières occurrences de chaque marque et son modèle. Pour rappel, les marques sont par ordre alphabétique dans le dataset. Les modèles de chaque marque sont classés du plus récent au plus ancien, comme ils le sont apparemment sur le site moviles, donc la première occurrence d'une marque m a son modèle le plus récent dès sa première apparition dans le dataset à l'index i.

In [21]:
first_o_indexes = p.first_occurrencesOf_observations(moviles_df, ['marque'])
moviles_df.iloc[first_o_indexes]

,marque,modele
0,apple,iphone 16
58,asus,zenfone 9
199,beafon,m6
258,blackview,a50 2022
362,crosscall,action-x5
387,cubot,p50
498,doogee,s98
647,doro,primo 368
738,google,pixel 9 pro fold
766,honor,x8c


On peut voir chaque marque et son modèle le plus récent dans notre dataset. Ainsi, nous pouvons savoir pour quelles marques doit-on compléter ses modèles les plus récents pour compléter notre référentiel.

In [22]:
moviles_df.loc[moviles_df['marque']=='astarry'].head(60)
#moviles_df.loc[moviles_df['marque']=='vivo'].tail(30)

,marque,modele


Ci-dessus, on sélectionne chaque marque et certains de ces modèles pour voir s'il est nécessaire de procéder à un ajout d'autres modèles ou non.

Mais pour s'assurer qu'on a laissé les modèles récents d'aucune marque de côté, on va vérifier, lors de cette nouvelle collecte, toutes les marques qu'on a, pour l'instant, dans cette liste :
    
    ['samsung', 'apple', 'xiaomi', 'blackview', 'reborn', 'oppo', 'google', 'oukitel', 
    'honor', 'oneplus', 'artfone', 'oscal', 'fossibot', 'cubot', 'nubia', 'doogee', 
    'doro', 'vivo', 'huawei', 'hotwav', 'crosscall', 'lagoona', 'olympia', 'generique', 
    'asus', 'motorola', 'beafon', 'realme', 'iiif150', 'viqee', 'fvh', 'xgody', 
    'rainbuvvy', 'astarry']  

Cette collecte est fait manuellement.

In [23]:
ADD_MODEL_DIC = os.getenv("ADD_MODEL_DIC")

En dehors d'ajouter les modèles les plus récents à ceux des marques du dataset moviles, nous avons exploré d'autres sources également, comme évoqué ci-dessus, pour avoir les modèles des marques qui n'y sont pas présentes. Ces sources sont en premier lieu les sites des marques, ensuite d'autres sources à chaque fois qu'il est nécessaire. Certaines marques, même si elles restent une exception, n'ont pas de site officiel. Dans ce cas, entre autres, je peux utiliser les sites e-commerces, entre autres sources, qui vendent leurs produits. 
    
Une autre manière simple et rapide d'avoir les modèles des marques est de les faire générer par une IA generative comme chatGPT, mais dans le cadre de ce portfolio, nous avons choisi la première solution qui vient d'être expliquée. En plus, même si elles seraient générées, elles ne seront pas dispensées de vérification pour voir s'il y a pas des incohérences, des modèles inventés pour des marques qui en réalité n'exitent pas, etc.
    
Après la collecte, j'ai instancié un dictionnaire et l'ai initialisé avec ces nouvelles données, ensuite, nous l'affectons à un dataframe pour leur éventuel nettoyage avant de les concatener au dataset moviles pour avoir un référentiel de modèles et leurs marques dont on a besoin dans la préparation de nos données e-commerces.

In [24]:
add_model_df = p.dic_to_df(ADD_MODEL_DIC, ['marque', 'modele'])
add_model_df = add_model_df.sort_values(by=['marque'], ignore_index=True)
add_model_df

,marque,modele
0,apple,iPhone 17 Pro
1,apple,iPhone 17 Pro Max
2,apple,iPhone Air
3,apple,iPhone 17
4,artfone,F20
...,...,...
494,xiaomi,Xiaomi 15 Ultra
495,xiaomi,Xiaomi 15T Pro
496,xiaomi,Xiaomi 15T
497,xiaomi,POCO F8 Pro


Comme nous pouvons le constater, nous avons maintenant le modèle iphone 17, par exemple, pour la marque Apple, alors qu'il n'est pas présent dans le dataset de moviles.  
Cependant, nous pouvons également constater que le nom de la marque est présent das certains modèles, nous allons les corriger plus tard.

**NB : La taille du dictionnaire peut changer au fur et à mesure qu'on ajoute de nouveaux modèles pour le mettre à jour, et par conséquent, la taille finale du référentiel. Donc, les tailles données, par la suite, le sont à un instant t.**

À présent, inspectons ces nouvelles données

In [25]:
# Inspecons nos nouvelles données
add_model_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 499 entries, 0 to 498
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   marque  499 non-null    object
 1   modele  499 non-null    object
dtypes: object(2)
memory usage: 7.9+ KB


In [26]:
add_model_df.describe()

,marque,modele
count,499,499
unique,31,483
top,rainbuvvy,ROG Phone 9
freq,58,3


Nous pouvons voir que nous avons 499 lignes, qui n'ont pas de valeurs nulles, avec 31 marques uniques et 483 modèles uniques.

Supprimons les doublons

In [27]:
add_model_df[add_model_df.duplicated()]

,marque,modele
18,asus,ROG Phone 8
23,asus,ROG Phone 9
28,asus,ROG Phone 7
29,asus,ROG Phone 9
147,fossibot,f105
194,hotwav,t7 pro
349,rainbuvvy,xs15 pro
367,rainbuvvy,q9
373,rainbuvvy,xs16
376,rainbuvvy,xs20


In [28]:
add_model_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 499 entries, 0 to 498
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   marque  499 non-null    object
 1   modele  499 non-null    object
dtypes: object(2)
memory usage: 7.9+ KB


In [29]:
add_model_df = add_model_df.drop_duplicates(ignore_index=True)
add_model_df.describe()

,marque,modele
count,484,484
unique,31,483
top,rainbuvvy,c80
freq,52,2


On peut voir maintenant que la taille du nouveau est de 484 lignes, donc les doublons ont été supprimés. 
Il y a le modèle c80 qui est dupliqué. Autrement dit, il y a deux marques différentes qui ont chacune un modèle qui porte le nom c80.
Quelles sont ces marques ?

In [30]:
add_model_df[add_model_df['modele'].duplicated(keep=False)]

,marque,modele
28,beafon,c80
270,oscal,c80


Ce sont les marques 'beafon' et 'oscal' qui ont chacune un modèle c80.

Dès lors qu'il n'y a plus d'observation dupliquée, faison une concatenation avec le dataset de moviles. Ensuite, faisons un nettoyage du dataset final, notamment la suppressions des éventuels doublons.

In [31]:
reference_df = pd.concat([moviles_df, add_model_df], ignore_index=True)
reference_df

,marque,modele
0,apple,iphone 16
1,apple,iphone 16 pro
2,apple,iphone 16 pro max
3,apple,iphone 16 plus
4,apple,iphone 15
...,...,...
5094,xiaomi,REDMI 15
5095,xiaomi,Xiaomi 15 Ultra
5096,xiaomi,Xiaomi 15T Pro
5097,xiaomi,Xiaomi 15T


In [32]:
reference_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5099 entries, 0 to 5098
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   marque  5099 non-null   object
 1   modele  5099 non-null   object
dtypes: object(2)
memory usage: 79.8+ KB


In [33]:
reference_df.describe()

,marque,modele
count,5099,5099
unique,32,4880
top,samsung,x20
freq,1471,5


On peut voit nous avons 5099 entrées. Le dataset est complet, il n'y a pas de valeurs nulles.
Nénamoins, nous avons des valeurs dupliqués. Nous pouvons également voir que les modèles n'ont pas le même format d'écriture. En effet, certains sont en minuscule, d'autres en majuscule et d'auttres encore en majuscule et minuscule.
    
Avant de traiter ces erreurs, voyons d'abord comment nos données sont concatener. Où se trouve par exemple les modèles récents de la marque 'apple' qu'on vient de concatener au dataset moviles afin de s'asurer qu'ils sont bien ajoutés dans le dataset final, même si dans l'affichage ci-dessus, on peut déjà voir que les derniers modèles de la marque 'xiaomi', qui ne sont pas présent dans le dataset moviles, sont bien présents dans le dataset concatené ?

In [34]:
reference_df.loc[reference_df['marque']=="apple"]

,marque,modele
0,apple,iphone 16
1,apple,iphone 16 pro
2,apple,iphone 16 pro max
3,apple,iphone 16 plus
4,apple,iphone 15
...,...,...
57,apple,iphone 16gb
4615,apple,iPhone 17 Pro
4616,apple,iPhone 17 Pro Max
4617,apple,iPhone Air


Nous pouvons voir qu'ils sont bien dans le dataset, mais comme c'est une concaténation, le fonction a placé le deuxième dataset à la fin du premier. Nous pouvons changé cela en triant le dataset.

Avant de faire cela, harmonisons les marques et les modèles en leur rendant tous minuscule et supprimant les éventuels de début et de fin

In [35]:
reference_df['marque'] = reference_df['marque'].str.lower().str.strip()
reference_df['modele'] = reference_df['modele'].str.lower().str.strip()
#sorted(reference_df['modele'].unique()[:60])
reference_df

,marque,modele
0,apple,iphone 16
1,apple,iphone 16 pro
2,apple,iphone 16 pro max
3,apple,iphone 16 plus
4,apple,iphone 15
...,...,...
5094,xiaomi,redmi 15
5095,xiaomi,xiaomi 15 ultra
5096,xiaomi,xiaomi 15t pro
5097,xiaomi,xiaomi 15t


Nous pouvons constater que dans les valeurs de certains modèles, nous avons la présence du nom de la marque, nous allons le supprimer avant de les trier

In [36]:
indexes = reference_df.index
p.delete_brand_label(reference_df, 'marque', 'modele', indexes)
reference_df['modele'].unique()

array(['iphone 16', 'iphone 16 pro', 'iphone 16 pro max', ..., '15t pro',
       '15t', 'poco f8 pro'], shape=(4856,), dtype=object)

In [37]:
reference_df = reference_df.sort_values(by=['marque','modele'], ignore_index=True)
reference_df.head()

,marque,modele
0,apple,iphone 11
1,apple,iphone 11 pro
2,apple,iphone 11 pro max
3,apple,iphone 12
4,apple,iphone 12 mini


Après la concatenation des deux datasets, nous venons de trier les observations, par ordre alphabétique, comme nous pouvons le constater dans l'affichage des données.  
Pour voir plus clair, affichons que les modèles de la première marque dans notre dataset qui est ici 'apple'. Ci-dessus, on a déjà vu qu'avant le tri, les modèles des nouvelles données sont ajoutés à partir de l'indice 4615, donc à la fin du dataset de moviles. 
Maintenant, revoyons où se situent-ils après le tri. Nous affichons dans le tableau suivant les modèles de la marque 'apple' de l'indice 20 à 30 (non inclus)

In [38]:
reference_df.loc[reference_df['marque']=="apple"][20:30]

,marque,modele
20,apple,iphone 16 plus
21,apple,iphone 16 pro
22,apple,iphone 16 pro max
23,apple,iphone 16gb
24,apple,iphone 17
25,apple,iphone 17 pro
26,apple,iphone 17 pro max
27,apple,iphone 3g 16gb
28,apple,iphone 3g 8gb
29,apple,iphone 3gs 16gb


On peut voir que les modèles 'iphone 17', 'iphone 17 pro', etc ont changé d'indices pour être parmi les premières indices. Ainsi, tous les modèles  de la marque 'apple' sont maintenant au même endroit dans les dataset.

À présent, nous allons vérifier les modèles de chaque marque et apporter les dernières corrections si besoin. Nous pouvions faire cette étape dès le début avant la fusion des datasets. Mais, après la fusion et la suppression des doublons et valeurs manquantes, nous pouvons faire ce nettoyage en une seule fois. Pour cela, affichons d'abord les marques distinctes que nous avons et les modèles uniques de chaque marque :

In [39]:
reference_df['marque'].unique()

array(['apple', 'artfone', 'astarry', 'asus', 'beafon', 'blackview',
       'crosscall', 'cubot', 'dooge', 'doogee', 'doro', 'fossibot', 'fvh',
       'google', 'honor', 'hotwav', 'huawei', 'iiif150', 'motorola',
       'nubia', 'olympia', 'oneplus', 'oppo', 'oscal', 'oukitel',
       'rainbuvvy', 'realme', 'samsung', 'viqee', 'vivo', 'xgody',
       'xiaomi'], dtype=object)

On peut constater que la marque "doogee" est écrit de deux manières 'dooge' et 'doogee', nous allons normaliser son écriture avec la dernière

In [40]:
reference_df['marque'] = reference_df['marque'].str.replace('dooge', 'doogee').str.strip()
reference_df['marque'] = reference_df['marque'].str.replace('doogeee', 'doogee').str.strip()
reference_df['marque'].unique()

array(['apple', 'artfone', 'astarry', 'asus', 'beafon', 'blackview',
       'crosscall', 'cubot', 'doogee', 'doro', 'fossibot', 'fvh',
       'google', 'honor', 'hotwav', 'huawei', 'iiif150', 'motorola',
       'nubia', 'olympia', 'oneplus', 'oppo', 'oscal', 'oukitel',
       'rainbuvvy', 'realme', 'samsung', 'viqee', 'vivo', 'xgody',
       'xiaomi'], dtype=object)

In [41]:
reference_df.loc[reference_df['marque']=='iiif150']['modele'].unique()
# reference_df.loc[reference_df['marque']=="samsung"]['modele'].unique()[1000:]

array(['action 15', 'action 15pro', 'action a5pro', 'air2 ultra', 'air3',
       'air3s', 'b2 pro', 'b3', 'b3 pro', 'b3c', 'raptor', 'raptor 5g',
       'raptor ltd'], dtype=object)

Ci-dessus, j'affiche les marques distincts, ensuite les modèles de chaque marque pour chercher ce qui est à nettoyer. À la suite de cette inspection, j'ai constitué deux listes de valeurs pour chaque marque. Ensuite, j'ai utilisé deux variables d'environnement qui sont des dictionnaires pour contenir les clés qui sont, pour l'un des dictionnaires, les marques sur lesquelles j'ai trouvé des choses à nettoyer, et les valeurs qui sont des listes de deux listes. La première liste contient les valeurs à remplacer, la deuxième contient les valeurs qui rempalcent, ou qui doivent être mises à la place des valeurs de la première liste. Les tailles de deux listes doivent être égales. 

Ainsi, il y a certaines valeurs à remplacer ou à supprimer dans tout le dataset, c'est ce que j'ai regroupé dans la cellule suivante dans les nettoyages généraux. J'ai les ai affectées dans une première variable d'environnement qui est G_DIC_TO_CLEAN avec comme clé des 'g' numérotés. Ensuite, il y a d'autres valeurs à remplacer ou à supprimer qui, elles, sont particulières à la marque. Ainsi, j'ai utilisé la marque comme clé. Je les ai affectées dans une seconde variable d'envrionnement qui est également un dictionnaire nommé DIC_TO_CLEAN. 

Une fois que les dictionnaires sont instanciés, j'ai implémenté une fonction qui prend un dataframe, la colonne à nettoyer, un dictionnaire et un paramètre des expressions régulières qui est par défaut False, et renvoie la colonne nettoyée. Elle appelle la fonction replace de pandas, qui est optimisée, pour faire les nettoyages. 

In [42]:
# Nettoyages généraux
# old_list = ['lte']
# new_list = ['lite']

# old_list_re = [r"\d+\s*gb"]
# new_list_re = [""]

# old_list_re = [r"\w+pro"]
# new_list_re = [""\w+\spro""]

# old_list_re = [r'\s*\+', r'\(', r'\)']
# new_list_re = [' plus', '', '']


# Nettoyages particuliers
# -- asus
# old_list = ["rog phone 8 pro edition", "rog phone 9 pro edition"]
# new_list = ["rog phone 8 pro", "rog phone 9 pro"]

# -- blackview
# old_list = ["bv6600 e"]
# new_list = ["bv6600e"]

# -- crosscall
# old_list = ["odyssey s1", "shark v2", "shark x3", "spider x1", "spider x3g", 
#             "spider x4", "trekker m1", "trekker x1", "trekker x3"]
# new_list = ["odyssey-s1", "shark-v2", "shark-x3", "spider-x1", "spider-x3g", 
#             "spider-x4", "trekker-m1", "trekker-x1", "trekker-x3"]

# -- cubot
# old_list = ["king kong"]
# new_list = ["kingkong"]

# -- doro
# old_list = ["330 handleeasy", "handle plus", "phone easy"]
# new_list = ["handleeasy 330", "handleplus", "phoneeasy"]

# -- huawei
# old_list = ["google nexus 6p",  "p8max"]
# new_list = ["nexus 6p", "p8 max"] 

# -- iiif150
# old_list = ['action 15pro', 'action a5pro']
# new_list = ['action 15 pro', 'action a5 pro']

# -- motorola
# old_list = ['c9 plus75', 'c9 plus80', "google nexus 6", "v.box"] 
# new_list = ['moto c', 'moto c plus', "nexus 6", "v100"]
# il n,existe pas de 'c9 plus75' et 'c9 plus80' pour motorola

# -- oppo
# old_list = ['find x lamborgini edition']
# new_list = ['find x lamborgini']

# -- oukitel
# old_list = ['wp100titan']
# new_list = ['wp100 titan']

# -- realme 
# old_list = ['c15 qualcomm edition']
# new_list = ['c15 qualcomm']

# -- samsung
# old_list = ['d900 ultra 12.9', 'd900i ultra 12.9', 'galaxy s 2 at&t']
# new_list = ['sgh-d900 ultra 12.9', 'sgh-d900i ultra 12.9', 'galaxy s ii']

# -- vivo
# old_list = ['nex dual display edition']
# new_list = ['nex dual display']# la marque l'a écrit ainsi aussi sans edition

# -- xiaomi
# old_list = ['mi-1s', 'mi-2', 'mi-2a', 'mi-2s', 'mi-3', 'redmi note 11se']
# new_list = ['mi 1s', 'mi 2', 'mi 2a', 'mi 2s', 'mi 3', 'redmi note 11 se']

In [43]:
precleaned_reference_df = reference_df.copy()

G_DIC_TO_CLEAN = os.getenv("G_DIC_TO_CLEAN")
DIC_TO_CLEAN = os.getenv("DIC_TO_CLEAN")

precleaned_reference_df['modele'] = p.clean_and_replace(precleaned_reference_df, 'modele', 
                                                        G_DIC_TO_CLEAN, regex=True)
precleaned_reference_df['modele'] = p.clean_and_replace(precleaned_reference_df, 'modele', DIC_TO_CLEAN)

--  g1
--  g2
--  g3
--  asus
--  blackview
--  crosscall
--  cubot
--  doro
--  huawei
--  iiif150
--  motorola
--  oppo
--  oukitel
--  realme
--  samsung
--  vivo
--  xiaomi


J'applique la fonction clean_and_replace, que j'ai implémenté, sur chacune les dictionnaires de valeurs à nettoyer. Cette fonction permet d'appliquer les nettoyages généraux à tous le dataset et ceux particuliers aux modèles de chaque marque.    
Après cet appel de la fonction, je revérifie les modèles de la marque pour voir si les nettoyages ont été effectués comme attendu.

In [44]:
# vérification après correction
precleaned_reference_df.loc[precleaned_reference_df['marque']=='iiif150']['modele'].unique()

array(['action 15', 'action 15 pro', 'action a5 pro', 'air2 ultra',
       'air3', 'air3s', 'b2 pro', 'b3', 'b3 pro', 'b3c', 'raptor',
       'raptor 5g', 'raptor ltd'], dtype=object)

On peut voir, par exemple, pour la marque 'iiif150' que les changements ont effectivement lieu, notamment 

        'action 15pro' -> 'action 15 pro' 
        'action a5pro' -> 'action a5 pro'
        
Ensuite, on vérifie qu'on a les mêmes marques dans le dataset avant et après ces nettoyages

In [45]:
reference_df['marque'].unique()

array(['apple', 'artfone', 'astarry', 'asus', 'beafon', 'blackview',
       'crosscall', 'cubot', 'doogee', 'doro', 'fossibot', 'fvh',
       'google', 'honor', 'hotwav', 'huawei', 'iiif150', 'motorola',
       'nubia', 'olympia', 'oneplus', 'oppo', 'oscal', 'oukitel',
       'rainbuvvy', 'realme', 'samsung', 'viqee', 'vivo', 'xgody',
       'xiaomi'], dtype=object)

In [46]:
precleaned_reference_df['marque'].unique()

array(['apple', 'artfone', 'astarry', 'asus', 'beafon', 'blackview',
       'crosscall', 'cubot', 'doogee', 'doro', 'fossibot', 'fvh',
       'google', 'honor', 'hotwav', 'huawei', 'iiif150', 'motorola',
       'nubia', 'olympia', 'oneplus', 'oppo', 'oscal', 'oukitel',
       'rainbuvvy', 'realme', 'samsung', 'viqee', 'vivo', 'xgody',
       'xiaomi'], dtype=object)

Nous avons les mêmes marques.  
Étant donné qu'on ne supprime pas de lignes, donc nous devons avoir le même nombre de lignes, vérifions si nous avons les mêmes observations pour chaque marque avant et après ces nettoyages 

In [47]:
len(reference_df.loc[reference_df['marque']=='apple']['modele'])

62

In [48]:
len(precleaned_reference_df.loc[precleaned_reference_df['marque']=='apple']['modele'])

62

On peut voir pour la marque 'apple', par exemple, que nous avons le même nombre de lignes avant et après les nettoyages.

Au cours de ces nettoyages, nous avons remarqué des lignes à supprimer. Après les vérications que nous venons de faire, nous allons ainsi sélectionner ces lignes à supprimer :

In [49]:
# fonepad note fhd6, live g500tg, 'watch gt 2 porsche design'
rows_to_delete = precleaned_reference_df.loc[(precleaned_reference_df['modele']=='fonepad note fhd6') |
                            (precleaned_reference_df['modele']=='live g500tg') |
                            (precleaned_reference_df['modele']=='watch gt 2 porsche design') |
                            (precleaned_reference_df['modele']=='iphone')]


In [50]:
precleaned_reference_df.drop(
    rows_to_delete.index,
    inplace=True
)
rows_to_delete

,marque,modele
23,apple,iphone
37,apple,iphone
53,apple,iphone
75,asus,fonepad note fhd6
78,asus,live g500tg
1430,huawei,watch gt 2 porsche design


Après avoir uniformisé le format d'écriture, effectué les nettoyages, normaliser les différents modèles, supprimé des valeurs qui n'étaient pas de modèles, traitons à présent les observations dupliquées.

In [51]:
# nombre de doublons dans le dataset avant les nettoyages
len(reference_df[reference_df.duplicated()])

19

In [52]:
# nombre de doublons dans le dataset après les nettoyages
len(precleaned_reference_df[precleaned_reference_df.duplicated()])

53

In [53]:
# Les observations dupliquées
# precleaned_reference_df[precleaned_reference_df.duplicated()]

In [54]:
# suppression des observations dupliquées
precleaned_reference_df.drop_duplicates(inplace=True, ignore_index=True)
# Après la suppression des doublons, nous affectons notre dataframe nettoyé dans 
# une nouvelle variable
cleaned_reference_df = precleaned_reference_df.copy()

Après la suppression des duplicats dans le dataset prénettoyé, inspectons à nouveau les deux dataset (avant et après les nettoyages apportés) pour avoir une vue d'ensemble sur les deux

In [55]:
reference_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5099 entries, 0 to 5098
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   marque  5099 non-null   object
 1   modele  5099 non-null   object
dtypes: object(2)
memory usage: 79.8+ KB


Avant les nettoyages ci-dessus, nous avions 5099 observations

In [56]:
cleaned_reference_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5040 entries, 0 to 5039
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   marque  5040 non-null   object
 1   modele  5040 non-null   object
dtypes: object(2)
memory usage: 78.9+ KB


Après les nettoyages, nous avons à présent 

In [57]:
reference_df.describe()

,marque,modele
count,5099,5099
unique,31,4856
top,samsung,x30
freq,1471,5


In [58]:
cleaned_reference_df.describe()

,marque,modele
count,5040,5040
unique,31,4815
top,samsung,x20
freq,1451,5


Avant et après les nettoyages, nous avons le même nombre de marques.

La taille de notre référentiel de modèles de marques est désormais passée de 5099 à 5040 après les nettoyages et la suppression des lignes dupliquées.
Cependant, des valeurs de modèle peuvent être dupliqués comme c'est le cas du modèle 'x20' qui est présent 5 fois dans la variable modèle. Cela est tout à fait compréhensible dans la mesure où deux marques peuvent avoir un même nom pour leurs modèles respectifs, comme expliqué ci-dessus. C'est pourquoi prendre comme clé (marque, modele), lors du tri, et les avoir tous dans le dataset est plus pertinent que de prendre seulement l'une des deux variables. Car si on prenait seulement les modèles sans les marques, par exemple, quand on cherchera dans nos données e-commerces si un modèle donné est présent ou non, on peut bien trouver que ce modèle est présent pour un produit, mais comment peut-on s'assurer que ce modèle n'est pas écrit par erreur par l'utilissateur ? Appartient-il réellement à cette marque ? 
C'est là où avoir un référenctiel des modèles et leurs marques à toute son importance. 

Enfin, nous pouvons exporter le référentiel dans un fichier csv

In [59]:
cleaned_reference_df.to_csv("../data/cleaned_data/reference_data.csv", index=False)
print("===== Données chargées")

===== Données chargées
